# Camera Extrinsic Calibration — Practice (OpenCV)

Execute the procedure from `extrinsic_calibration.md` using **OpenCV's real API**.
No real images needed — the notebook is self-contained, generating **synthetic checkerboard correspondences**.

1. Estimate intrinsic parameters (K, distortion) with `calibrateCamera`
2. Estimate extrinsic parameters `[R|t]` for one camera with `solvePnP`
3. Estimate relative `R,T` between two cameras with `stereoCalibrate`
4. Rectification (R1, R2, P1, P2, Q) with `stereoRectify`
5. Disparity → depth (reprojection via Q)

> Understand the theory first in `extrinsic_calibration_demo.ipynb` (numpy).

In [ ]:
# %pip install opencv-python numpy matplotlib
import cv2, numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True); np.random.seed(0)
print('OpenCV', cv2.__version__)

---

## 0. Synthetic Data Generation

Place a checkerboard (9×6 inner corners) in multiple poses and project it with known K and distortion to create `imagePoints`.
`cv2.projectPoints` handles projection including distortion.

In [ ]:
PATTERN=(9,6); SQUARE=0.025          # 25mm squares
IMG_W,IMG_H=1280,720; IMG_SIZE=(IMG_W,IMG_H)

# 3D points in board coordinate system (z=0 plane)
objp=np.zeros((PATTERN[0]*PATTERN[1],3),np.float32)
objp[:,:2]=np.mgrid[0:PATTERN[0],0:PATTERN[1]].T.reshape(-1,2)*SQUARE

# True intrinsic parameters (target of estimation)
K_gt=np.array([[800,0,640],[0,800,360]],np.float64)
K_gt=np.array([[800,0,640],[0,800,360],[0,0,1]],np.float64)
dist_gt=np.array([0.05,-0.08,0.001,0.0005,0.0],np.float64)  # k1,k2,p1,p2,k3

def random_board_pose():
    rvec=np.array([np.random.uniform(-0.4,0.4),np.random.uniform(-0.4,0.4),
                   np.random.uniform(-0.3,0.3)])
    tvec=np.array([np.random.uniform(-0.15,0.15),np.random.uniform(-0.1,0.1),
                   np.random.uniform(0.6,1.0)])           # 0.6-1.0m forward
    return rvec.reshape(3,1),tvec.reshape(3,1)

N_VIEWS=18
poses=[random_board_pose() for _ in range(N_VIEWS)]
objpoints=[]; imgpoints=[]
for rvec,tvec in poses:
    img,_=cv2.projectPoints(objp,rvec,tvec,K_gt,dist_gt)
    objpoints.append(objp.copy())
    imgpoints.append(img.reshape(-1,1,2).astype(np.float32))
print(f'Generated correspondences for {N_VIEWS} poses. {len(objp)} points per pose')

---

## 1. Intrinsic Calibration `calibrateCamera`

Estimate K and distortion from correspondences, then compare against ground truth (Zhang's method).

In [ ]:
ret,K_est,dist_est,rvecs,tvecs=cv2.calibrateCamera(
    objpoints,imgpoints,IMG_SIZE,None,None)
print('Reprojection RMS error:',round(ret,5),'px')
print('K ground truth:\n',K_gt)
print('K estimated:\n',K_est)
print('dist ground truth:',dist_gt)
print('dist estimated:',dist_est.ravel())

---

## 2. Extrinsic Parameters `solvePnP`

Given known K, estimate `[R|t]` for one view (§4). Compare against the true pose.

In [ ]:
i=0
ok,rvec_est,tvec_est=cv2.solvePnP(objpoints[i],imgpoints[i],K_est,dist_est)
R_est,_=cv2.Rodrigues(rvec_est)
R_gt,_=cv2.Rodrigues(poses[i][0])
ang=np.rad2deg(np.arccos((np.trace(R_gt.T@R_est)-1)/2))
print('Rotation error:',round(float(ang),4),'deg')
print('t ground truth:',poses[i][1].ravel(),' / estimated:',tvec_est.ravel())
print('Camera center C=-R^T t:', (-R_est.T@tvec_est).ravel())

# Reprojection error
rep,_=cv2.projectPoints(objpoints[i],rvec_est,tvec_est,K_est,dist_est)
err=np.linalg.norm(rep.reshape(-1,2)-imgpoints[i].reshape(-1,2),axis=1)
print('Reprojection error mean:',round(err.mean(),4),'px')

---

## 3. Stereo Calibration `stereoCalibrate`

Estimate relative pose `R,T` between two cameras (§5.3). Create synthetic correspondences by observing the same board simultaneously from both cameras.

In [ ]:
# True relative pose (cam1 -> cam2): mainly horizontal baseline + slight convergence
R_rel_gt,_=cv2.Rodrigues(np.array([0.0,np.deg2rad(-3),0.0]))
T_rel_gt=np.array([[0.10],[0.0],[0.0]])     # Baseline 10cm
K2_gt=K_gt.copy(); dist2_gt=dist_gt.copy()

objpts=[]; imgL=[]; imgR=[]
for _ in range(N_VIEWS):
    rvec,tvec=random_board_pose()
    Rb,_=cv2.Rodrigues(rvec)
    # Board pose in cam2: X_c2 = R_rel(R_b X + t_b) + T_rel
    Rb2=R_rel_gt@Rb; tb2=R_rel_gt@tvec+T_rel_gt
    rvec2,_=cv2.Rodrigues(Rb2)
    iL,_=cv2.projectPoints(objp,rvec,tvec,K_gt,dist_gt)
    iR,_=cv2.projectPoints(objp,rvec2,tb2,K2_gt,dist2_gt)
    objpts.append(objp.copy())
    imgL.append(iL.reshape(-1,1,2).astype(np.float32))
    imgR.append(iR.reshape(-1,1,2).astype(np.float32))

flags=cv2.CALIB_FIX_INTRINSIC   # Treat intrinsics as known, estimate only relative pose
ret,K1,d1,K2,d2,R_rel,T_rel,E,F=cv2.stereoCalibrate(
    objpts,imgL,imgR,K_gt,dist_gt,K2_gt,dist2_gt,IMG_SIZE,flags=flags)
print('Stereo RMS:',round(ret,5),'px')
ang=np.rad2deg(np.arccos((np.trace(R_rel_gt.T@R_rel)-1)/2))
print('Relative rotation error:',round(float(ang),4),'deg')
print('T ground truth:',T_rel_gt.ravel(),' / estimated:',T_rel.ravel())

---

## 4. Rectification `stereoRectify`

Obtain `R1, R2` (per-camera rectification rotations), `P1, P2` (new projection matrices), and `Q` (reprojection matrix) (§6).

In [ ]:
R1,R2,P1,P2,Q,roi1,roi2=cv2.stereoRectify(
    K1,d1,K2,d2,IMG_SIZE,R_rel,T_rel,flags=cv2.CALIB_ZERO_DISPARITY,alpha=0)
print('P1=\n',P1)
print('P2=\n',P2,'  ← P2[0,3] = -f*baseline (horizontal offset)')
f_rect=P1[0,0]; baseline=-P2[0,3]/P2[0,0]
print(f'Rectified f={f_rect:.2f}px, baseline={baseline*100:.2f}cm')
print('Q=\n',Q)

In [ ]:
# Remap maps for rectification (combines distortion correction + alignment)
map1x,map1y=cv2.initUndistortRectifyMap(K1,d1,R1,P1,IMG_SIZE,cv2.CV_32FC1)
map2x,map2y=cv2.initUndistortRectifyMap(K2,d2,R2,P2,IMG_SIZE,cv2.CV_32FC1)
print('Remap map generation complete:',map1x.shape)

# Validation: transform correspondences to rectified coordinates and check if v matches left/right
ptsL=imgL[0].reshape(-1,2); ptsR=imgR[0].reshape(-1,2)
rL=cv2.undistortPoints(ptsL.reshape(-1,1,2),K1,d1,R=R1,P=P1).reshape(-1,2)
rR=cv2.undistortPoints(ptsR.reshape(-1,1,2),K2,d2,R=R2,P=P2).reshape(-1,2)
dv=np.abs(rL[:,1]-rR[:,1])
print('Mean vertical misalignment after rectification |v_L - v_R|:',round(dv.mean(),4),'px (≈0 means horizontal alignment)')

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(rL[:,0],rL[:,1],c='b',s=10,label='Left')
plt.scatter(rR[:,0],rR[:,1],c='r',s=10,label='Right')
for i in range(0,len(rL),5): plt.axhline(rL[i,1],color='gray',lw=0.3)
plt.gca().invert_yaxis(); plt.legend()
plt.title('After rectification: corresponding points aligned on same row'); plt.xlabel('u'); plt.ylabel('v'); plt.show()

---

## 5. Disparity → Depth (Reprojection via Q)

Recover 3D (depth) from disparity `d` after rectification using reprojection matrix `Q` (§6.3).

In [ ]:
# Reconstruct 3D from disparity d = u_L - u_R using Q
disp=rL[:,0]-rR[:,0]
# Q: [X,Y,Z,W]^T = Q [u, v, d, 1]^T,  3D = (X,Y,Z)/W
homog=np.c_[rL[:,0],rL[:,1],disp,np.ones(len(rL))]
XYZW=(Q@homog.T).T
XYZ=XYZW[:,:3]/XYZW[:,3:4]
print('Disparity → depth Z range:',round(XYZ[:,2].min(),3),'-',round(XYZ[:,2].max(),3),'m')

# Also verify consistency with formula Z=f*B/d
Z_formula=f_rect*baseline/disp
print('Max difference between Z from Q and f*B/d:',round(np.abs(XYZ[:,2]-Z_formula).max(),5),'m')

---

## 6. Hands-on Exercises

1. **Increase distortion**: Increase k1, k2 in `dist_gt` and observe `calibrateCamera` estimation accuracy
2. **Reduce number of poses**: Set `N_VIEWS=4` and observe instability (§8 degeneracy)
3. **Change baseline**: Set `T_rel_gt` to 0.05/0.30m and calculate depth resolution difference with `Z=fB/d`
4. **Add noise**: Add Gaussian noise to imagePoints and observe changes in RMS error and extrinsic parameter error
5. **Real images**: Your own checkerboard images → `cv2.findChessboardCorners` → same flow as this notebook
6. **nuScenes**: Read `calibrated_sensor` rotation/translation and reproduce camera-LiDAR projection

> Log observations in #daily-action-log to reflect in the daily review's "Learning & Growth" section.